# Case
**Creates binary labels in raster format**

The labels are either based on a provided text- or Esri-shapefile.<br>
To spatially align the labels with other data sets, a template raster (base raster) can be provided.


In [ ]:

import geopandas as gpd

import sys
if sys.version_info < (3, 9):
    from importlib_resources import files
else:
    from importlib.resources import files

from beak.experimental.conversions import create_geodataframe_from_points, create_binary_raster
from beak.experimental.io import load_raster, load_dataset


# Load data

**User definitions**

In [ ]:
# Paths
BASE_PATH = files("beak.data")
PATH_DATACUBE = BASE_PATH / "LAWLEY22" / "RAW" / "2021_Table04_Datacube.csv"
PATH_SHAPEFILE = BASE_PATH / "raw" / "mineral_deposits" / "critical_mineral_deposits" / "critical_mineral_deposits.shp"
PATH_BASE_RASTER = BASE_PATH / "BASE_RASTERS" / "EPSG_4326_RES_0_02_US_CANADA_LAWLEY_DC.tif"

# Columns for x and y coordinates
LONG_COL = "Longitude_EPSG4326"
LAT_COL = "Latitude_EPSG4326"

# Reference coordinate system code
EPSG_CODE = 4326

# Rows to read from the datacube
NROWS = None

# Filter datacube by region
REGIONS =["United States of America", "Canada"]
COL_REGION_FILTER = "Country_Majority"

# Select relevant columns to be used as labels
LABELS = "Training_MVT_Deposit=='Present' | Training_MVT_Occurrence=='Present'"

# Create Labels

## From textfile or data set

In [ ]:
# Load data
data = load_dataset(PATH_DATACUBE, nrows=NROWS)
data = data[data[COL_REGION_FILTER].isin(REGIONS)]

# Create geodataframe from points
gdf = create_geodataframe_from_points(data, LONG_COL, LAT_COL, EPSG_CODE)

# Load base raster
base_raster = load_raster(PATH_BASE_RASTER)

# Output path
out_path = "output_from_textfile.tif"

# Save raster and labels array
labels_array = create_binary_raster(gdf, base_raster, query=LABELS, all_touched=False, same_shape=True, fill_negatives=True, out_file=out_path)

## From shapefile

### Points

In [ ]:
# Load shapefile and create geodataframe
gdf = gpd.read_file(PATH_SHAPEFILE)

# Load base raster
base_raster = load_raster(PATH_BASE_RASTER)

# Output path
out_path = "output_from_shapefile.tif"

# Save raster and labels array
labels_array, meta = create_binary_raster(gdf, 
                                          base_raster, 
                                          query="DepType == 'S-R-V tungsten'", 
                                          all_touched=False, 
                                          same_shape=True, 
                                          fill_negatives=True, 
                                          out_file=out_path,
                                          return_meta=True)